# Entry notebook: "point at a folder and run"

Contract: the user types in no values — no temperature, no selections, no
number of sites. Only the path to the system config may be edited, and even
that is overridden by the `PCM2_CONFIG` environment variable for a
non-interactive run. The notebook **recomputes nothing** — it reads the run
artifacts: the feature builder WRITES its output, so a seemingly harmless
cell would otherwise corrupt a published run. If an artifact is missing, the
cell says so instead of failing: the corresponding step has not been run yet.

In [ ]:
import json, os, sys
from pathlib import Path
HERE = Path.cwd() if (Path.cwd() / 'src' / 'pcm2').exists() else Path.cwd().parent
sys.path.insert(0, str(HERE / 'src'))
from pcm2 import config as pcm2_config
from pcm2.runtime import run_dir

CONFIG = os.environ.get('PCM2_CONFIG', str(HERE / 'configs' / 'gramicidin.yaml'))
cfg = pcm2_config.load(CONFIG)
ROOT = run_dir(cfg)
print('system:', cfg['system.id'], '| run folder:', ROOT)

def artifact(*parts):
    p = ROOT.joinpath(*parts)
    if not p.exists():
        print(f'{p.relative_to(ROOT)} is missing — that step has not been run yet; run: '
              f'python -m pcm2.run all --config {CONFIG}')
        return None
    return p

## What autodetect measured and what it refused

In [ ]:
p = artifact('autodetect', 'report.json')
if p:
    rep = json.loads(p.read_text())
    for key, ent in sorted(rep.get('origins_after', {}).items()):
        print(f"{ent['origin']:9s} {key}: {ent.get('basis','')[:100]}")
    for r in rep.get('refusals', []):
        print('REFUSED:', r['key'], '—', r['message'])

## Events and labels

In [ ]:
p = artifact('events', 'summary.json')
if p:
    s = json.loads(p.read_text())
    for k, v in s['replicas'].items():
        print(k, '— own events:', v['n_own'], '| provided:', v['n_provided'],
              '| left-censored at start:', v['left_censored_at_start'])
p = artifact('labels', 'summary.json')
if p:
    s = json.loads(p.read_text())
    for k, v in s['per_replica'].items():
        print(k, {kk: vv for kk, vv in v.items() if kk.startswith('tau')})

## Per-frame answer: probability, verdict, mechanism

In [ ]:
import pandas as pd
p = artifact('train', 'answers.parquet')
if p:
    ans = pd.read_parquet(p)
    display(ans.head(12))
    print('READY fraction:', float((ans["ready"] == "READY").mean()))
    print('mechanism axes:')
    print(ans['mechanism_axis'].value_counts())

## Metrics and figures

In [ ]:
p = artifact('train', 'evaluation.json')
if p:
    ev = json.loads(p.read_text())
    primary = str(cfg['labels.primary_tau_ps'])
    for arm, taus in sorted(ev['per_arm'].items()):
        pooled = taus.get(primary, {}).get('pooled', {})
        if pooled.get('defined'):
            print(f"{arm:28s} AP={pooled['ap']:.4f} "
                  f"(x{pooled['ratio_to_chance']:.2f} over chance)")
from IPython.display import Image, display
p = artifact('figures', 'fig_pr_primary.png')
if p:
    display(Image(str(p)))

## Manual overrides

The cell below is empty by default and is needed only if autodetect refused and
named a key: declare it in the config (`origins`, with a basis) and re-run the
corresponding step.